# 🫀 AI-Powered Heart Murmur Detection: BiLSTM Training & Hugging Face Export
### End-to-End Pipeline: Kaggle Dataset ➔ Audio Signal Processing ➔ PyTorch BiLSTM ➔ Hugging Face Model Hub

This notebook trains a deep learning Bidirectional LSTM (BiLSTM) model with temporal attention pooling to detect heart murmurs and arrhythmias from acoustic phonocardiogram (PCG) recordings.

**Workflow Outline:**
1. **Environment Setup & GPU Check**
2. **Kaggle Authentication & Dataset Download** (`abdallahaboelkhair/heartbeat-sound`)
3. **Data Preprocessing & Audio Feature Extraction** (MFCCs, Spectral Contrast, Chroma, RMS)
4. **Stratified Dataset Splitting & PyTorch DataLoaders**
5. **BiLSTM Neural Network Architecture**
6. **Training with Early Stopping & Learning Rate Scheduler**
7. **Clinical Performance Evaluation (Confusion Matrix & Classification Report)**
8. **Export Model to Hugging Face Hub**

## Step 1: Install Dependencies & Check GPU

In [ ]:
# Install necessary audio processing and Hugging Face packages
!pip install -q kaggle librosa soundfile torch torchvision torchaudio huggingface_hub scikit-learn matplotlib seaborn tqdm

import os
import json
import glob
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

## Step 2: Connect Kaggle & Download Heartbeat Sound Dataset

> **How to get your `kaggle.json` token:**
> 1. Go to [kaggle.com/settings](https://www.kaggle.com/settings)
> 2. Scroll to the **API** section and click **Create New Token** (downloads `kaggle.json`).
> 3. Run the cell below to upload `kaggle.json` directly to Colab.

In [ ]:
#@title Step 2: Kaggle Authentication & Download
import os
from google.colab import files

# Enter Kaggle Access Token or upload kaggle.json
KAGGLE_ACCESS_TOKEN = "" #@param {type:"string"}

os.makedirs("/root/.kaggle", exist_ok=True)
if KAGGLE_ACCESS_TOKEN.strip():
    with open("/root/.kaggle/access_token", "w") as f:
        f.write(KAGGLE_ACCESS_TOKEN.strip() + "\n")
    os.chmod("/root/.kaggle/access_token", 0o600)
    print("Configured Kaggle access token!")
elif os.path.exists("/root/.kaggle/kaggle.json") or os.path.exists("/root/.kaggle/access_token"):
    print("Existing Kaggle credentials found.")
else:
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        import shutil
        shutil.move(fn, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Download and unzip dataset
DATASET_NAME = "abdallahaboelkhair/heartbeat-sound"
!kaggle datasets download -d {DATASET_NAME} -p /content/dataset --unzip
print("Dataset downloaded and unzipped successfully!")


## Step 3: Dataset Discovery & Class Label Parsing

In [ ]:
# Inspect dataset directory
dataset_root = "/content/dataset"
print("Contents of dataset:", os.listdir(dataset_root))

# Locate CSV files or audio directories
data_records = []

# The Pascal challenge dataset typically has set_a and set_b
for set_name in ["set_a", "set_b"]:
    csv_path = os.path.join(dataset_root, f"{set_name}.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            fname = row.get("fname", "")
            label = row.get("label", "")
            if pd.isna(label) or str(label).strip() == "":
                continue
            # Find full audio path
            full_path = os.path.join(dataset_root, set_name, fname)
            if not os.path.exists(full_path):
                # Try without set prefix
                matches = glob.glob(os.path.join(dataset_root, "**", fname), recursive=True)
                full_path = matches[0] if matches else None
            
            if full_path and os.path.exists(full_path):
                data_records.append({"filepath": full_path, "raw_label": str(label).strip()})

# Fallback search if CSV didn't match all files
if len(data_records) == 0:
    for wav_file in glob.glob(os.path.join(dataset_root, "**", "*.wav"), recursive=True):
        parent = os.path.basename(os.path.dirname(wav_file))
        filename = os.path.basename(wav_file).lower()
        if "murmur" in filename or "murmur" in parent.lower():
            lbl = "murmur"
        elif "extra" in filename or "extra" in parent.lower():
            lbl = "extrahls"
        elif "normal" in filename or "normal" in parent.lower():
            lbl = "normal"
        else:
            continue
        data_records.append({"filepath": wav_file, "raw_label": lbl})

df_all = pd.DataFrame(data_records)
print(f"Total valid audio samples discovered: {len(df_all)}")
print("Raw Label counts:\n", df_all["raw_label"].value_counts())

# Standardize into 3 main diagnostic classes: Normal, Murmur, Extrasystole
def standardize_label(lbl):
    lbl = str(lbl).lower()
    if "murmur" in lbl:
        return "Murmur"
    elif "extra" in lbl or "arrhythmia" in lbl:
        return "Extrasystole"
    elif "normal" in lbl:
        return "Normal"
    return None

df_all["class"] = df_all["raw_label"].apply(standardize_label)
df_clean = df_all.dropna(subset=["class"]).reset_index(drop=True)

print("\nCleaned Diagnostic Classes:")
print(df_clean["class"].value_counts())

# Visualize class distribution
plt.figure(figsize=(6, 3))
sns.countplot(data=df_clean, x="class", palette="viridis")
plt.title("Dataset Heart Sound Class Distribution")
plt.show()

## Step 4: Audio Preprocessing & Feature Extraction

We extract **60 acoustic features** per time frame:
- 40 Mel-Frequency Cepstral Coefficients (MFCCs)
- 7 Spectral Contrast bands (distinguishing turbulent murmur frequencies)
- 12 Chroma pitch bins
- 1 RMS Energy value

In [ ]:
TARGET_SR = 4000         # 4000 Hz captures heart auscultation frequency band
DURATION = 5.0           # 5-second fixed window
TARGET_SAMPLES = int(TARGET_SR * DURATION)
N_FFT = 512
HOP_LENGTH = 128
N_MFCC = 40

CLASSES = ["Normal", "Murmur", "Extrasystole"]
class2idx = {c: i for i, c in enumerate(CLASSES)}

def extract_audio_features(filepath):
    # Load & resample
    y, sr = librosa.load(filepath, sr=TARGET_SR, mono=True)
    
    # Peak normalize
    max_val = np.max(np.abs(y))
    if max_val > 1e-6:
        y = y / max_val
    else:
        y = np.zeros_like(y)
        
    # Pad or truncate
    if len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)), mode="constant")
    else:
        y = y[:TARGET_SAMPLES]
        
    # Feature extraction
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    try:
        contrast = librosa.feature.spectral_contrast(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, fmin=50.0, n_bands=6)
    except Exception:
        contrast = np.zeros((7, mfcc.shape[1]), dtype=np.float32)
    chroma = librosa.feature.chroma_stft(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)
    
    min_t = min(mfcc.shape[1], contrast.shape[1], chroma.shape[1], rms.shape[1])
    stacked = np.vstack([
        mfcc[:, :min_t],
        contrast[:, :min_t],
        chroma[:, :min_t],
        rms[:, :min_t]
    ])
    
    # Standardize per band
    mean = np.mean(stacked, axis=1, keepdims=True)
    std = np.std(stacked, axis=1, keepdims=True) + 1e-6
    norm_feat = (stacked - mean) / std
    return norm_feat.T.astype(np.float32)  # (seq_len, feature_dim)

print("Extracting features from all audio samples...")
X_data, y_data = [], []
for _, row in tqdm(df_clean.iterrows(), total=len(df_clean)):
    try:
        feat = extract_audio_features(row["filepath"])
        X_data.append(feat)
        y_data.append(class2idx[row["class"]])
    except Exception as e:
        continue

X_data = np.array(X_data)
y_data = np.array(y_data)
print(f"Extracted feature tensor shape: {X_data.shape} (N, seq_len, feature_dim)")

## Step 5: Stratified Dataset Split & PyTorch DataLoader

In [ ]:
# Split: 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, test_size=0.30, stratify=y_data, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train samples: {len(X_train)} | Val samples: {len(X_val)} | Test samples: {len(X_test)}")

class HeartSoundDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 16
train_loader = DataLoader(HeartSoundDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(HeartSoundDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(HeartSoundDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

## Step 6: Define BiLSTM with Attention Pooling

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        weights = self.attention(x)  # (B, T, 1)
        weights = F.softmax(weights, dim=1)
        context = torch.sum(weights * x, dim=1)  # (B, hidden_dim)
        return context

class HeartMurmurLSTM(nn.Module):
    def __init__(self, input_dim=60, hidden_dim=128, num_layers=2, dropout=0.3, num_classes=3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        lstm_out_dim = hidden_dim * 2
        self.pool = AttentionPooling(lstm_out_dim)
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(lstm_out_dim),
            nn.Linear(lstm_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        proj = self.input_proj(x)
        lstm_out, _ = self.lstm(proj)
        context = self.pool(lstm_out)
        logits = self.classifier(context)
        return logits

model = HeartMurmurLSTM(input_dim=X_data.shape[2], hidden_dim=128, num_layers=2, dropout=0.3, num_classes=len(CLASSES)).to(device)
print(model)

## Step 7: Train Model with Class-Weighted Loss & Cosine LR

In [ ]:
# Compute class weights to handle natural class imbalance
class_counts = np.bincount(y_train)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / np.sum(class_weights)
weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

EPOCHS = 35
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    # Training
    model.train()
    running_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(by)
    
    train_loss = running_loss / len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            val_loss += loss.item() * len(by)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == by).sum().item()
            
    val_loss = val_loss / len(val_loader.dataset)
    val_acc = correct / len(val_loader.dataset)
    scheduler.step()
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save best model checkpoint locally
        torch.save({"state_dict": model.state_dict()}, "best_heart_murmur_lstm.pt")
        
    if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.1f}%")

# Plot Training Curves
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.title("Loss Curves")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["val_acc"], label="Val Accuracy", color="green")
plt.title("Validation Accuracy")
plt.legend()
plt.show()

## Step 8: Comprehensive Test Set Evaluation

In [ ]:
# Load best checkpoint for evaluation
checkpoint = torch.load("best_heart_murmur_lstm.pt")
model.load_state_dict(checkpoint["state_dict"])
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        logits = model(bx)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(by.numpy())

# Classification Report
print("=== Clinical Classification Report ===")
print(classification_report(all_targets, all_preds, target_names=CLASSES))

# Confusion Matrix Heatmap
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("Confusion Matrix - Heart Sound Auscultation")
plt.xlabel("Predicted Category")
plt.ylabel("Ground Truth")
plt.show()

## Step 9: Export & Push Model to Hugging Face Model Hub

> **How to get your Hugging Face Access Token:**
> 1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
> 2. Click **Create new token**, select **Write** permission, and copy the token.
> 3. Paste it into the prompt below.

In [ ]:
#@title Step 9: Hugging Face Export
import os, json
from huggingface_hub import login, HfApi, create_repo

HF_TOKEN = "" #@param {type:"string"}
HF_REPO_ID = "machhakiran/heart-murmur-bilstm" #@param {type:"string"}

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("Please enter your Hugging Face Write Token:")
    login()

api = HfApi(token=HF_TOKEN if HF_TOKEN else None)
create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
print(f"Hugging Face repository verified at: https://huggingface.co/{HF_REPO_ID}")

# 1. Save model checkpoint
final_checkpoint = {
    "state_dict": model.state_dict(),
    "metadata": {
        "config": {
            "input_dim": 60,
            "hidden_dim": 128,
            "num_layers": 2,
            "dropout": 0.3,
            "bidirectional": True,
            "num_classes": 3
        },
        "classes": CLASSES
    }
}
torch.save(final_checkpoint, "heart_murmur_lstm.pt")

# 2. Save config.json
config_data = {
    "model_type": "HeartMurmurLSTM",
    "classes": CLASSES,
    "sample_rate": TARGET_SR,
    "duration_seconds": DURATION,
    "feature_dimension": 60
}
with open("config.json", "w") as f:
    json.dump(config_data, f, indent=2)

# 3. Create Model Card (README.md)
model_card = f"""---
tags:
- audio-classification
- medical-ai
- cardiology
- phonocardiogram
- pytorch
license: mit
---
# 🫀 AI Heart Murmur & Arrhythmia Detection Model

Trained in Google Colab on Kaggle Heartbeat Sound Challenge (abdallahaboelkhair/heartbeat-sound).

## Target Classes
- Normal: Regular S1/S2 lub-dub rhythm
- Murmur: Systolic / diastolic murmur turbulence
- Extrasystole: Premature cardiac contraction / arrhythmia
"""
with open("README.md", "w") as f:
    f.write(model_card)

# Upload files to Hugging Face
print("Uploading model files to Hugging Face Hub...")
api.upload_file(path_or_fileobj="heart_murmur_lstm.pt", path_in_repo="heart_murmur_lstm.pt", repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj="config.json", path_in_repo="config.json", repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj="README.md", path_in_repo="README.md", repo_id=HF_REPO_ID)

print(f"\n Successfully deployed Colab model to Hugging Face Model Hub!")
print(f"Model URL: https://huggingface.co/{HF_REPO_ID}")
